# LexData — Pipeline de datos · Nicho Familiar · **v8**

**Cambios v8 respecto a v7:**
- 🔧 INMLCF/Policía: `$where` con columnas correctas → filtra en API, no lee 600k registros
- 🔧 Helper `build_where_depto()` / `build_where_años()` para SOQL seguro
- ♻️ Fuente 3: `hurto_personas` reemplazado por `inasistencia_alimentaria` (Fiscalía)
  → delito directo del nicho familiar vs. proxy de criminalidad urbana
- 🔧 ICBF: fallback automático al catálogo Socrata cuando el ID primario está 404
- 🔧 PESOS_IVF: `hurto_total` → `inasistencia_total`

**Fuentes activas:**
| # | Dataset | ID | Estado |
|---|---|---|---|
| 1 | INMLCF VIF Forense | `ers2-kerr` | ✅ |
| 2 | Policía SIEDCO VIF | `vuyt-mqpw` + `kmnf-h6r5` | ✅ |
| 3 | Inasistencia Alimentaria — Fiscalía | `hf4m-4hbq` | ✅ |
| 4 | Comisarías Ley 2126 | `7tuu-upb2` | ✅ |
| 5 | ICBF medidas protección | `wpqv-gzbz` + fallback | ⚠️ auto-fallback |


## Sección 1 — Configuración

In [1]:
import requests

url = "https://www.datos.gov.co/resource/ers2-kerr.json"
params = {"$limit": 5}

r = requests.get(url, params=params)
print(r.status_code)
print(r.json()[:2])

200
[{'id': '1', 'a_o_del_hecho': '2015', 'sexo_de_la_victima': 'Hombre', 'grupo_de_edad_quinquenal': '(10 a 14)', 'grupo_mayor_menor_de_edad': 'a) Menor de Edad (<18 Años)', 'grupo_de_edad_judicial': '(10 a 13)', 'ciclo_vital': '(12 a 17) Adolescencia', 'pais_de_nacimiento': 'Colombia', 'escolaridad': 'Básica primaria', 'estado_civil': 'Soltero (a)', 'tipo_de_discapacidad': 'Ninguna', 'pertenencia_etnica': 'Sin información', 'orientacion_sexual': 'No Sabe / No Informa', 'identidad_de_genero': 'No Sabe / No Informa', 'transgenero': 'No Sabe / No Informa', 'pertenencia_grupal': 'Sin información', 'mes_del_hecho': 'Diciembre', 'dia_del_hecho': 'martes', 'rango_de_hora_del_hecho_x_3_horas': '(18:00 a 20:59)', 'codigo_dane_municipio': '11001', 'municipio_del_hecho_dane': 'Bogotá, D.C.', 'departamento_del_hecho_dane': 'Bogotá, D.C.', 'codigo_dane_departamento': '11', 'localidad_del_hecho': 'Rafael Uribe Uribe', 'zona_del_hecho': 'Cabecera municipal', 'escenario_del_hecho': 'Calle (Autopista

In [2]:
import requests
import pandas as pd
import numpy as np
import time
import os
import unicodedata
from datetime import datetime

# ── Configuración general ─────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(
    os.path.expanduser("~"),
    "OneDrive", "Escritorio", "CARPETAS",
    "septimo semestre", "data thinking",
    "2segunda entrega", "LexData", "data_judicial"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_URL = "https://www.datos.gov.co"

DATASETS = {
    "vif_inmlcf":               "ers2-kerr",   # INMLCF VIF forense 2015-2024
    "vif_policia":              "vuyt-mqpw",   # Policía SIEDCO VIF
    "vif_policia_ext":          "kmnf-h6r5",   # Policía VIF por municipio
    "inasistencia_alimentaria": "hf4m-4hbq",   # Fiscalía — inasistencia alimentaria
    "comisarias_directorio":    "7tuu-upb2",   # Directorio comisarías Ley 2126
    "comisarias_icbf":          "wpqv-gzbz",   # ICBF medidas protección (fallback auto)
}

YEARS = list(range(2025, 2019, -1))  # [2025, 2024, 2023, 2022, 2021, 2020]

YEAR_COLS = ["a_o", "anio", "year", "a__o", "vigencia", "año", "a_o_del_hecho"]

DEPARTAMENTOS_FILTRO = ["VALLE DEL CAUCA"]

# ── Pesos IVF — nicho familiar ────────────────────────────────────────────────
# VIF (INMLCF + Policía): indicador más directo y grave del ciclo
# Alimentos: consecuencia civil de ruptura familiar (proxy 0.75×VIF hasta tener CSJ)
# Medidas ICBF: intervención institucional por vulnerabilidad infantil
# Inasistencia alimentaria: delito penal directo del ciclo familiar (Fiscalía)
PESOS_IVF = {
    "vif_total":                0.40,
    "alimentos_familia_total":  0.30,
    "medidas_proteccion_total": 0.20,
    "inasistencia_total":       0.10,   # antes: hurto_total — ahora delito familiar real
}

DANE_POB_2024 = {
    "CALI": 2237030, "PALMIRA": 311063, "BUENAVENTURA": 436665, "TULUA": 221048,
    "JAMUNDI": 167441, "YUMBO": 118397, "GUADALAJARA DE BUGA": 122601, "CANDELARIA": 103840,
    "CARTAGO": 138001, "FLORIDA": 63458, "EL CERRITO": 57248, "PRADERA": 54283,
    "SEVILLA": 44218, "ZARZAL": 44100, "GUACARI": 31420, "DAGUA": 35800,
    "CALIMA": 22100, "CAICEDONIA": 28900, "BUGALAGRANDE": 22000, "GINEBRA": 20800,
    "LA UNION": 36000, "ROLDANILLO": 38000, "YOTOCO": 17900, "ANDALUCIA": 19800,
    "SAN PEDRO": 16500, "ALCALA": 17200, "LA CUMBRE": 13100, "ANSERMANUEVO": 16300,
    "RESTREPO": 16000, "VIJES": 14700, "RIOFRIO": 17400, "OBANDO": 13700,
    "TRUJILLO": 17200, "LA VICTORIA": 14000, "BOLIVAR": 22100, "TORO": 17900,
    "ULLOA": 6800, "VERSALLES": 8700, "EL DOVIO": 9100, "EL AGUILA": 9300,
    "EL CAIRO": 8800, "ARGELIA": 10400,
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (LexData-Scraper/8.0; nicho-familiar)",
    "Accept": "application/json",
}

print(f"✅ LexData Nicho Familiar v8")
print(f"   Período: {min(YEARS)}–{max(YEARS)}")
print(f"   Departamento: {DEPARTAMENTOS_FILTRO}")
print(f"   Fuentes: {len(DATASETS)} datasets del ciclo familiar")
print(f"   Output: {OUTPUT_DIR}")


✅ LexData Nicho Familiar v8
   Período: 2020–2025
   Departamento: ['VALLE DEL CAUCA']
   Fuentes: 6 datasets del ciclo familiar
   Output: C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\data_judicial


In [3]:
def buscar_dataset_icbf():
    """
    Busca automáticamente un dataset alternativo en Socrata
    relacionado con comisarías / ICBF.
    """
    print("🔎 Buscando dataset alternativo de ICBF en Socrata...")

    url = "https://api.us.socrata.com/api/catalog/v1"
    params = {
        "q": "comisarias familia violencia intrafamiliar colombia",
        "domains": "www.datos.gov.co",
        "limit": 5
    }

    try:
        res = requests.get(url, params=params, timeout=10).json()
        resultados = res.get("results", [])

        for ds in resultados:
            nombre = ds["resource"]["name"]
            did = ds["resource"]["id"]
            print(f"   → {nombre} | ID: {did}")

        if resultados:
            nuevo_id = resultados[0]["resource"]["id"]
            print(f"✅ Usando dataset alternativo: {nuevo_id}")
            return nuevo_id

    except Exception as e:
        print(f"⚠️ Error buscando dataset alternativo: {e}")

    return None

## Sección 2 — Utilidades de red

In [4]:
# ── Mapeo de aliases de columnas → nombre estándar ───────────────────────────
COL_ALIAS = {
    "municipio": [
        "municipio_del_hecho_dane", "municipio_hecho", "municipio_del_hecho",
        "nombre_municipio", "nom_municipio", "despacho_municipio", "nombre_1",
    ],
    "departamento": [
        "departamento_del_hecho_dane", "departamento_hecho", "departamento_del_hecho",
        "nombre_departamento", "nom_departamento", "despacho_departamento",
    ],
    "anio": [
        "a_o_del_hecho", "a_o", "a__o", "vigencia", "año", "year",
    ],
}


def build_endpoint(dataset_id: str) -> str:
    return f"{BASE_URL}/resource/{dataset_id}.json"


def build_where_depto(col_name: str, deptos: list = None) -> str:
    """Construye cláusula SOQL $where para filtrar departamento en la API.
    Más eficiente que traer todos los registros y filtrar en pandas.
    Uso: params['$where'] = build_where_depto('departamento_del_hecho_dane')
    """
    deptos = deptos or DEPARTAMENTOS_FILTRO
    vals = ", ".join(f"'{d}'" for d in deptos)
    return f"upper({col_name}) in ({vals})"


def build_where_años(col_name: str, years: list = None) -> str:
    """Construye cláusula SOQL $where para filtrar años en la API.
    col_name: nombre de la columna de año en el dataset (string o numérica).
    """
    years = years or YEARS
    vals = ", ".join(f"'{y}'" for y in years)
    return f"{col_name} in ({vals})"


def socrata_get(session, dataset_id: str, params: dict, max_pages: int = 200, max_retries: int = 3) -> list:
    """
    GET paginado con retry + backoff exponencial.
    max_pages=200 → hasta 200k registros por dataset.
    Devuelve lista vacía si el dataset no existe o falla persistentemente.
    """
    endpoint = build_endpoint(dataset_id)
    PAGE_SIZE = 1000
    results, offset, page = [], 0, 0

    while page < max_pages:
        p = {**params, "$limit": PAGE_SIZE, "$offset": offset}
        batch = None

        for attempt in range(max_retries):
            try:
                r = session.get(endpoint, headers=HEADERS, params=p, timeout=30)
                if r.status_code == 400:
                    print(f"    ⚠️ 400 Bad Request — query SOQL inválida")
                    print(f"       $where problemático: {params.get('$where', 'N/A')[:150]}")
                    return []
                if r.status_code in (403, 404):
                    print(f"    ✗ Dataset {dataset_id} → HTTP {r.status_code}")
                    return []
                r.raise_for_status()
                batch = r.json()
                break
            except requests.exceptions.HTTPError as e:
                wait = 2 ** attempt
                print(f"    ⚠ HTTP {e} — reintento {attempt+1}/{max_retries} en {wait}s")
                time.sleep(wait)
            except requests.exceptions.ConnectionError:
                wait = 2 ** attempt
                print(f"    ⚠ Error de red — reintento {attempt+1}/{max_retries} en {wait}s")
                time.sleep(wait)
            except Exception as e:
                print(f"    ✗ Error inesperado: {e}")
                return results

        if batch is None:
            print(f"    ✗ {dataset_id} no responde tras {max_retries} intentos")
            break
        if not batch:
            break

        results.extend(batch)
        if len(batch) < PAGE_SIZE:
            break
        offset += PAGE_SIZE
        page += 1
        time.sleep(0.4)

    return results


def normalizar_columnas(df: pd.DataFrame, fuente_log: str = "") -> pd.DataFrame:
    """Renombra automáticamente columnas al estándar (municipio, departamento, anio)."""
    rename_map = {}
    for std_name, aliases in COL_ALIAS.items():
        if std_name not in df.columns:
            for alias in aliases:
                if alias in df.columns:
                    rename_map[alias] = std_name
                    tag = f"[{fuente_log}] " if fuente_log else ""
                    print(f"    🔄 {tag}'{alias}' → '{std_name}'")
                    break
    if rename_map:
        df = df.rename(columns=rename_map)
    return df


def detectar_col_fecha(df: pd.DataFrame) -> str | None:
    """Detecta la columna de año por nombre o contenido."""
    year_cols_lower = [y.lower() for y in YEAR_COLS]
    for col in df.columns:
        if col.lower() in year_cols_lower:
            return col
    for col in df.select_dtypes(include=["object"]).columns:
        sample = df[col].dropna().head(20)
        if len(sample) > 0 and sample.str.match(r'^\d{4}$').mean() > 0.8:
            return col
    for col in df.select_dtypes(include=["datetime64"]).columns:
        return col
    return None


def normalizar_texto(t) -> str:
    nfkd = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in nfkd if not unicodedata.combining(c)).upper().strip()


def filtrar_depto(df: pd.DataFrame) -> pd.DataFrame:
    """Filtra por DEPARTAMENTOS_FILTRO sobre la columna 'departamento' (ya estandarizada)."""
    if not DEPARTAMENTOS_FILTRO or "departamento" not in df.columns:
        return df
    deptos_norm = [normalizar_texto(d) for d in DEPARTAMENTOS_FILTRO]
    n_antes = len(df)
    df = df[df["departamento"].apply(normalizar_texto).isin(deptos_norm)].copy()
    print(f"    🗺  Filtro depto: {n_antes} → {len(df)} registros {DEPARTAMENTOS_FILTRO}")
    return df


def filtrar_años(df: pd.DataFrame, col_año: str) -> pd.DataFrame:
    if col_año not in df.columns:
        return df
    df = df.copy()
    df[col_año] = pd.to_numeric(df[col_año], errors="coerce")
    n_antes = len(df)
    df = df[df[col_año].isin(YEARS)].copy()
    print(f"    📅 Filtro años {min(YEARS)}–{max(YEARS)}: {n_antes} → {len(df)} registros")
    return df


def detectar_col_año(df: pd.DataFrame) -> str | None:
    for col in df.columns:
        if col.lower() in [y.lower() for y in YEAR_COLS]:
            return col
    return None


def safe_df(registros: list, fuente: str, dataset_id: str) -> pd.DataFrame:
    df = pd.DataFrame(registros)
    if not df.empty:
        df["fuente"] = fuente
        df["url_dataset"] = build_endpoint(dataset_id)
        df = df.drop_duplicates()
    return df


def inspect_dataset(session, dataset_id: str) -> dict:
    endpoint = build_endpoint(dataset_id)
    try:
        r = session.get(endpoint, headers=HEADERS, params={"$limit": 2}, timeout=15)
        if r.status_code in (403, 404):
            return {"disponible": False, "error": f"HTTP {r.status_code}", "columnas": [], "muestra": []}
        r.raise_for_status()
        data = r.json()
        columnas = list(data[0].keys()) if data else []
        return {"disponible": True, "columnas": columnas, "muestra": data[:1]}
    except Exception as e:
        return {"disponible": False, "error": str(e), "columnas": [], "muestra": []}


print("✅ Utilidades v8 cargadas")
print("   → build_where_depto() / build_where_años() para SOQL seguro")
print("   → normalizar_columnas() con COL_ALIAS centralizado")


✅ Utilidades v8 cargadas
   → build_where_depto() / build_where_años() para SOQL seguro
   → normalizar_columnas() con COL_ALIAS centralizado


## Sección 3 — Diagnóstico de datasets

In [5]:
print("=" * 60)
print("DIAGNÓSTICO — LexData Nicho Familiar v6")
print("=" * 60)

session_diag = requests.Session()
DATASETS_DISPONIBLES = {}

# Test de conectividad
try:
    r_test = session_diag.get(
        build_endpoint("ers2-kerr"), headers=HEADERS,
        params={"$limit": 1}, timeout=15
    )
    r_test.raise_for_status()
    print("✅ Conectividad OK — datos.gov.co accesible")
except Exception as e:
    print(f"⚠️ Problema de conectividad: {e}")

print()

for nombre, did in DATASETS.items():
    resultado = inspect_dataset(session_diag, did)
    DATASETS_DISPONIBLES[nombre] = resultado["disponible"]
    icono = "✅" if resultado["disponible"] else "❌"
    print(f"{icono} [{nombre}] ID: {did}")
    if resultado["disponible"] and resultado["columnas"]:
        print(f"   Columnas ({len(resultado['columnas'])}): {', '.join(resultado['columnas'][:6])}")
    elif not resultado["disponible"]:
        print(f"   Error: {resultado.get('error', 'desconocido')}")
    print()

activos = sum(DATASETS_DISPONIBLES.values())
print("=" * 60)
print(f"Datasets activos: {activos}/{len(DATASETS)}")
if activos < len(DATASETS):
    caidos = [k for k, v in DATASETS_DISPONIBLES.items() if not v]
    print(f"⚠️ Caídos: {caidos} — el pipeline continúa sin ellos")
print("=" * 60)

DIAGNÓSTICO — LexData Nicho Familiar v6
✅ Conectividad OK — datos.gov.co accesible

✅ [vif_inmlcf] ID: ers2-kerr
   Columnas (36): id, a_o_del_hecho, sexo_de_la_victima, grupo_de_edad_quinquenal, grupo_mayor_menor_de_edad, grupo_de_edad_judicial

✅ [vif_policia] ID: vuyt-mqpw
   Columnas (8): departamento, municipio, codigo_dane, armas_medios, fecha_hecho, genero

✅ [vif_policia_ext] ID: kmnf-h6r5
   Columnas (8): departamento, municipio, codigo_dane, armas_medios, fecha_hecho, genero

❌ [inasistencia_alimentaria] ID: hf4m-4hbq
   Error: HTTP 404

✅ [comisarias_directorio] ID: 7tuu-upb2
   Columnas (15): c_digo_dane_departamento, nombre, c_digo_dane_municipio, nombre_1, tipo_municipio_isla_rea_no, categoria_municipio

❌ [comisarias_icbf] ID: wpqv-gzbz
   Error: HTTP 404

Datasets activos: 4/6
⚠️ Caídos: ['inasistencia_alimentaria', 'comisarias_icbf'] — el pipeline continúa sin ellos


## Sección 4 — Scraping por fuente

### Fuente 1 — INMLCF Violencia Intrafamiliar

In [6]:
def scrape_vif_inmlcf(session, years: list) -> pd.DataFrame:
    """
    INMLCF — Violencia Intrafamiliar Forense (ers2-kerr)
    FIX v8: $where con columnas REALES verificadas en diagnóstico:
      departamento_del_hecho_dane, a_o_del_hecho
    Esto filtra en la API y evita leer 600k registros nacionales.
    """
    DID = DATASETS["vif_inmlcf"]
    if not DATASETS_DISPONIBLES.get("vif_inmlcf", False):
        print("  ⚠ vif_inmlcf no disponible — omitiendo")
        return pd.DataFrame()

    print("[INMLCF VIF] Extrayendo datos con filtro API...")

    # ✅ $where con nombres de columna reales (verificados en diagnóstico)
    where_depto = build_where_depto("departamento_del_hecho_dane")
    where_años  = build_where_años("a_o_del_hecho", years)
    params = {"$where": f"({where_depto}) AND ({where_años})"}
    print(f"  → $where: {params['$where'][:100]}")

    raw = socrata_get(session, DID, params)
    print(f"  → {len(raw)} registros crudos")
    if not raw:
        # Fallback: sin filtro año (solo depto) por si el formato de año difiere
        print("  → Reintentando sin filtro de año...")
        params_fb = {"$where": where_depto}
        raw = socrata_get(session, DID, params_fb)
        print(f"  → {len(raw)} registros (fallback sin año)")
    if not raw:
        return pd.DataFrame()

    df = safe_df(raw, "INMLCF — VIF Forense", DID)
    print(f"  → Columnas: {list(df.columns)[:6]}")

    # Normalizar alias → municipio, departamento, anio
    df = normalizar_columnas(df, fuente_log="INMLCF")

    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].apply(normalizar_texto)
        df = filtrar_depto(df)   # segunda guardia por si el $where no cubrió variantes

    if "municipio" in df.columns:
        df["municipio"] = df["municipio"].apply(normalizar_texto)
    else:
        df["municipio"] = "SIN_DATO"

    # Filtrar año en pandas (segunda guardia)
    if "anio" not in df.columns:
        col_año = detectar_col_fecha(df)
        if col_año:
            df[col_año] = pd.to_numeric(df[col_año], errors="coerce")
            df = df.rename(columns={col_año: "anio"})
    df = filtrar_años(df, "anio")

    df["tipo_ciclo"] = "VIF"
    df["cantidad"] = pd.to_numeric(df.get("cantidad", 1), errors="coerce").fillna(1) \
                     if "cantidad" in df.columns else 1

    print(f"  ✅ INMLCF VIF: {len(df):,} registros · municipios: {df['municipio'].nunique()}")
    return df


session_main = requests.Session()
df_vif_inmlcf = scrape_vif_inmlcf(session_main, YEARS)

if not df_vif_inmlcf.empty:
    ruta = os.path.join(OUTPUT_DIR, "lexdata_vif_inmlcf.csv")
    df_vif_inmlcf.to_csv(ruta, index=False)
    print(f"  💾 Guardado: {ruta}")

df_vif_inmlcf.head(3)


[INMLCF VIF] Extrayendo datos con filtro API...
  → $where: (upper(departamento_del_hecho_dane) in ('VALLE DEL CAUCA')) AND (a_o_del_hecho in ('2025', '2024', '
  → 5856 registros crudos
  → Columnas: ['id', 'a_o_del_hecho', 'sexo_de_la_victima', 'grupo_de_edad_quinquenal', 'grupo_mayor_menor_de_edad', 'grupo_de_edad_judicial']
    🔄 [INMLCF] 'municipio_del_hecho_dane' → 'municipio'
    🔄 [INMLCF] 'departamento_del_hecho_dane' → 'departamento'
    🔄 [INMLCF] 'a_o_del_hecho' → 'anio'
    🗺  Filtro depto: 5856 → 5856 registros ['VALLE DEL CAUCA']
    📅 Filtro años 2020–2025: 5856 → 5856 registros
  ✅ INMLCF VIF: 5,856 registros · municipios: 41
  💾 Guardado: C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\data_judicial\lexdata_vif_inmlcf.csv


,id,anio,sexo_de_la_victima,grupo_de_edad_quinquenal,grupo_mayor_menor_de_edad,grupo_de_edad_judicial,ciclo_vital,pais_de_nacimiento,escolaridad,estado_civil,...,diagnostico_topografico_de_la_lesion_no_fatal,sexo_del_agresor,presunto_agresor_detallado,factor_desencadenante_de_la_agresion,dias_de_incapacidad_medicolegal,pueblo_indigena,fuente,url_dataset,tipo_ciclo,cantidad
0,138092,2020,Mujer,(25 a 29),b) Mayores de Edad (>18 años),(25 a 28),(18 a 28) Juventud,Colombia,Educación básica primaria,Soltero (a),...,Politraumatismo,Hombre,Hermano (a),Consumo de alcohol y/o sustancias psicoactivas,1 a 30,No aplica,INMLCF — VIF Forense,https://www.datos.gov.co/resource/ers2-kerr.json,VIF,1
1,138093,2020,Hombre,(40 a 44),b) Mayores de Edad (>18 años),(40 a 44),(29 a 59) Adultez,Colombia,Educación media o secundaria alta,Unión libre,...,Politraumatismo,Hombre,Primo (a),Consumo de alcohol y/o sustancias psicoactivas,1 a 30,No aplica,INMLCF — VIF Forense,https://www.datos.gov.co/resource/ers2-kerr.json,VIF,1
2,138094,2020,Mujer,(55 a 59),b) Mayores de Edad (>18 años),(55 a 59),(29 a 59) Adultez,Colombia,Educación básica primaria,Viudo (a),...,Politraumatismo,Hombre,Hermano (a),Consumo de alcohol y/o sustancias psicoactivas,1 a 30,No aplica,INMLCF — VIF Forense,https://www.datos.gov.co/resource/ers2-kerr.json,VIF,1


### Fuente 2 — Policía SIEDCO VIF

In [7]:
def scrape_vif_policia(session, years: list) -> pd.DataFrame:
    """
    Policía SIEDCO — VIF (vuyt-mqpw + kmnf-h6r5)
    FIX v8: $where=upper(departamento) — columna real confirmada en diagnóstico.
    Fecha en DD/MM/YYYY → dayfirst=True para año correcto.
    """
    todos = []

    for nombre_ds in ["vif_policia", "vif_policia_ext"]:
        DID = DATASETS[nombre_ds]
        if not DATASETS_DISPONIBLES.get(nombre_ds, False):
            print(f"  ⚠ {nombre_ds} no disponible — omitiendo")
            continue

        print(f"[Policía VIF — {nombre_ds}] Extrayendo con filtro API...")

        # ✅ $where con columna real 'departamento' (confirmada en diagnóstico)
        where_depto = build_where_depto("departamento")
        params = {"$where": where_depto}
        print(f"  → $where: {where_depto}")

        raw = socrata_get(session, DID, params)
        print(f"  → {len(raw)} registros · cols: {list(raw[0].keys()) if raw else []}")
        todos.extend(raw)
        time.sleep(0.5)

    if not todos:
        return pd.DataFrame()

    df = pd.DataFrame(todos).drop_duplicates()

    df = normalizar_columnas(df, fuente_log="Policía VIF")

    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].astype(str).str.upper().str.strip()
        df = filtrar_depto(df)   # segunda guardia

    if "municipio" in df.columns:
        df["municipio"] = df["municipio"].apply(normalizar_texto)
    else:
        df["municipio"] = "SIN_DATO"

    # ✅ dayfirst=True — fechas colombianas DD/MM/YYYY
    col_fecha = next((c for c in ["fecha_hecho", "fecha"] if c in df.columns), None)
    if col_fecha:
        df["anio"] = pd.to_datetime(
            df[col_fecha], dayfirst=True, errors="coerce"
        ).dt.year.astype("Int64")
        df = filtrar_años(df, "anio")
    elif "anio" in df.columns:
        df = filtrar_años(df, "anio")

    df["tipo_ciclo"] = "VIF"
    df["fuente"]     = "Policía SIEDCO — VIF"
    df["cantidad"]   = 1

    print(f"  ✅ Policía VIF: {len(df):,} registros")
    return df


df_vif_policia = scrape_vif_policia(session_main, YEARS)

if not df_vif_policia.empty:
    ruta = os.path.join(OUTPUT_DIR, "lexdata_vif_policia.csv")
    df_vif_policia.to_csv(ruta, index=False)
    print(f"  💾 Guardado: {ruta}")

df_vif_policia.head(3)


[Policía VIF — vif_policia] Extrayendo con filtro API...
  → $where: upper(departamento) in ('VALLE DEL CAUCA')
  → 0 registros · cols: []
[Policía VIF — vif_policia_ext] Extrayendo con filtro API...
  → $where: upper(departamento) in ('VALLE DEL CAUCA')
  → 0 registros · cols: []


""


### Fuente 3 — Inasistencia Alimentaria (Fiscalía — delito directo del nicho familiar)

In [8]:
# IDs a intentar para inasistencia alimentaria — Fiscalía
# hf4m-4hbq: dataset principal (Fiscalía SPOA — inasistencia alimentaria)
# Si falla, busca en catálogo automáticamente
INASISTENCIA_IDS = ["hf4m-4hbq"]


def buscar_inasistencia_catalogo(session) -> str | None:
    """Busca en el catálogo Socrata un dataset de inasistencia alimentaria."""
    url = "https://api.us.socrata.com/api/catalog/v1"
    queries = [
        "inasistencia alimentaria fiscalia colombia",
        "inasistencia alimentaria colombia",
    ]
    for q in queries:
        try:
            r = session.get(url, params={"q": q, "domains": "www.datos.gov.co", "limit": 5},
                            timeout=12)
            resultados = r.json().get("results", [])
            for ds in resultados:
                nombre = ds["resource"]["name"]
                did = ds["resource"]["id"]
                print(f"    📋 Catálogo: '{nombre}' | ID: {did}")
            if resultados:
                return resultados[0]["resource"]["id"]
        except Exception as e:
            print(f"    ⚠ Error catálogo: {e}")
    return None


def scrape_inasistencia_alimentaria(session, years: list) -> pd.DataFrame:
    """
    Fiscalía — Inasistencia Alimentaria.
    Delito penal directo del ciclo familiar: padre/madre que incumple obligación
    económica → indicador de ruptura económica con consecuencias familiares.
    Reemplaza 'hurto a personas' que medía criminalidad urbana genérica.
    """
    # Determinar DID disponible
    DID = None
    if DATASETS_DISPONIBLES.get("inasistencia_alimentaria", False):
        DID = DATASETS["inasistencia_alimentaria"]
    else:
        print("  ⚠ inasistencia_alimentaria no disponible — buscando alternativa...")
        DID = buscar_inasistencia_catalogo(session)
        if DID:
            # Verificar que el ID encontrado responde
            info = inspect_dataset(session, DID)
            if not info["disponible"]:
                DID = None

    if not DID:
        print("  ✗ No se encontró dataset de inasistencia alimentaria")
        return pd.DataFrame()

    print(f"[Inasistencia Alimentaria] Extrayendo — ID: {DID}...")

    # Detectar columna de departamento disponible para $where
    info = inspect_dataset(session, DID)
    col_depto = next(
        (c for c in info.get("columnas", []) if "departamento" in c.lower()),
        None
    )
    print(f"  → Columnas detectadas (muestra): {info.get('columnas', [])[:8]}")

    if col_depto:
        params = {"$where": build_where_depto(col_depto)}
        print(f"  → $where: {params['$where']}")
    else:
        # Sin $where — descarga completa y filtra en pandas
        params = {}
        print("  → Sin $where (columna departamento no detectada) — filtrando en pandas")

    raw = socrata_get(session, DID, params)
    print(f"  → {len(raw)} registros crudos")
    if not raw:
        return pd.DataFrame()

    df = pd.DataFrame(raw).drop_duplicates()
    df = normalizar_columnas(df, fuente_log="Inasistencia")

    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].astype(str).str.upper().str.strip()
        df = filtrar_depto(df)

    if "municipio" in df.columns:
        df["municipio"] = df["municipio"].apply(normalizar_texto)
    else:
        df["municipio"] = "SIN_DATO"

    # Año
    if "anio" not in df.columns:
        col_año = detectar_col_fecha(df)
        if col_año:
            col_fecha = next((c for c in ["fecha_hecho", "fecha"] if c in df.columns), None)
            if col_fecha and col_fecha == col_año:
                df["anio"] = pd.to_datetime(
                    df[col_fecha], dayfirst=True, errors="coerce"
                ).dt.year.astype("Int64")
            else:
                df[col_año] = pd.to_numeric(df[col_año], errors="coerce")
                df = df.rename(columns={col_año: "anio"})
    if "anio" in df.columns:
        df = filtrar_años(df, "anio")

    df["tipo_ciclo"] = "INASISTENCIA"
    df["fuente"] = "Fiscalía — Inasistencia Alimentaria"
    if "cantidad" not in df.columns:
        df["cantidad"] = 1
    else:
        df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(1)

    print(f"  ✅ Inasistencia alimentaria: {len(df):,} registros · municipios: {df['municipio'].nunique()}")
    return df


df_inasistencia = scrape_inasistencia_alimentaria(session_main, YEARS)

if not df_inasistencia.empty:
    ruta = os.path.join(OUTPUT_DIR, "lexdata_inasistencia_alimentaria.csv")
    df_inasistencia.to_csv(ruta, index=False)
    print(f"  💾 Guardado: {ruta}")

df_inasistencia.head(3)


  ⚠ inasistencia_alimentaria no disponible — buscando alternativa...
    📋 Catálogo: 'Avance_Atencion_PNIS' | ID: v4pt-rnn9
    📋 Catálogo: 'Reporte Hurto por Modalidades Policía Nacional' | ID: 9vha-vh9n
    📋 Catálogo: 'Reporte Hurto por Modalidades Policía Nacional' | ID: d4fr-sbn2
    📋 Catálogo: 'Amenazas Policía Nacional de Colombia' | ID: meew-mguv
    📋 Catálogo: 'Reporte Hurto por Modalidades Policía Nacional' | ID: 6sqw-8cg5
[Inasistencia Alimentaria] Extrayendo — ID: v4pt-rnn9...
  → Columnas detectadas (muestra): ['divipola_municipal', 'departamento', 'municipio', 'pagos_asistencia_alimentaria', 'asistencia_t_cnica_integral', 'autosostenimiento_y_seguridad', 'proyectos_productivos_pp_corto', 'proyectos_productivos_pp_largo']
  → $where: upper(departamento) in ('VALLE DEL CAUCA')
  → 3 registros crudos
    🗺  Filtro depto: 3 → 3 registros ['VALLE DEL CAUCA']
  ✅ Inasistencia alimentaria: 3 registros · municipios: 3
  💾 Guardado: C:\Users\Acer\OneDrive\Escritorio\CARPETAS\sep

,divipola_municipal,departamento,municipio,pagos_asistencia_alimentaria,asistencia_t_cnica_integral,autosostenimiento_y_seguridad,proyectos_productivos_pp_corto,proyectos_productivos_pp_largo,recolectores,fecha_de_corte,tipo_ciclo,fuente,cantidad
0,76100,VALLE DEL CAUCA,BOLIVAR,263,269,260,218,244,15,2026-02-27T00:00:00.000,INASISTENCIA,Fiscalía — Inasistencia Alimentaria,1
1,76233,VALLE DEL CAUCA,DAGUA,413,423,405,355,398,40,2026-02-27T00:00:00.000,INASISTENCIA,Fiscalía — Inasistencia Alimentaria,1
2,76250,VALLE DEL CAUCA,EL DOVIO,150,152,143,124,142,6,2026-02-27T00:00:00.000,INASISTENCIA,Fiscalía — Inasistencia Alimentaria,1


### Fuente 4 — ICBF Medidas de Protección

In [9]:
# IDs a intentar para ICBF/comisarías — en orden de preferencia
ICBF_IDS_FALLBACK = [
    "wpqv-gzbz",   # ID original (puede estar 404)
    "sgf5-3gg7",   # alternativo conocido — medidas restablecimiento derechos
    "t2uk-ntbr",   # violencia sexual intrafamiliar INMLCF (segundo proxy)
]


def buscar_icbf_catalogo(session) -> str | None:
    """Busca en catálogo Socrata un dataset ICBF/comisarías activo."""
    url = "https://api.us.socrata.com/api/catalog/v1"
    queries = [
        "medidas proteccion comisarias familia colombia",
        "icbf medidas restablecimiento derechos colombia",
        "comisarias familia violencia intrafamiliar colombia",
    ]
    for q in queries:
        try:
            r = session.get(url, params={"q": q, "domains": "www.datos.gov.co", "limit": 5},
                            timeout=12)
            resultados = r.json().get("results", [])
            for ds in resultados:
                nombre = ds["resource"]["name"]
                did = ds["resource"]["id"]
                print(f"    📋 Catálogo: '{nombre}' | ID: {did}")
                # Verificar que responde antes de retornar
                info = inspect_dataset(session, did)
                if info["disponible"]:
                    print(f"    ✅ ID {did} responde — usando como alternativa")
                    return did
        except Exception as e:
            print(f"    ⚠ Error catálogo: {e}")
    return None


def scrape_icbf_medidas(session, years: list) -> pd.DataFrame:
    """
    ICBF — Medidas de Protección Comisarías.
    FIX v8: intenta múltiples IDs → búsqueda en catálogo → fallback vacío con mensaje.
    """
    # 1. Intentar IDs conocidos en orden
    DID = None
    for did_candidato in ICBF_IDS_FALLBACK:
        info = inspect_dataset(session, did_candidato)
        if info["disponible"]:
            DID = did_candidato
            print(f"  ✅ ICBF: usando ID {DID}")
            break
        else:
            print(f"  ✗ ICBF ID {did_candidato} → {info.get('error', 'no disponible')}")

    # 2. Si ningún ID conocido funciona, buscar en catálogo
    if not DID:
        print("  → Buscando alternativa en catálogo Socrata...")
        DID = buscar_icbf_catalogo(session)

    if not DID:
        print("  ✗ ICBF: no se encontró dataset activo — dimensión medidas_proteccion = 0")
        return pd.DataFrame()

    print(f"[ICBF Medidas] Extrayendo — ID: {DID}...")

    # Detectar columna departamento para $where
    info = inspect_dataset(session, DID)
    col_depto = next(
        (c for c in info.get("columnas", []) if "departamento" in c.lower()),
        None
    )
    print(f"  → Columnas (muestra): {info.get('columnas', [])[:8]}")

    if col_depto:
        params = {"$where": build_where_depto(col_depto)}
        print(f"  → $where: {params['$where']}")
    else:
        params = {}

    raw = socrata_get(session, DID, params)
    print(f"  → {len(raw)} registros crudos")
    if not raw:
        return pd.DataFrame()

    df = pd.DataFrame(raw).drop_duplicates()
    df = normalizar_columnas(df, fuente_log="ICBF")

    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].astype(str).str.upper().str.strip()
        df = filtrar_depto(df)

    if "municipio" in df.columns:
        df["municipio"] = df["municipio"].apply(normalizar_texto)
    else:
        df["municipio"] = "SIN_DATO"

    if "anio" not in df.columns:
        col_año = detectar_col_fecha(df)
        if col_año:
            df[col_año] = pd.to_numeric(df[col_año], errors="coerce")
            df = df.rename(columns={col_año: "anio"})
    if "anio" in df.columns:
        df = filtrar_años(df, "anio")

    df["tipo_ciclo"] = "MEDIDA_ICBF"
    df["fuente"] = f"ICBF — Medidas de Protección ({DID})"
    df["cantidad"] = pd.to_numeric(df.get("cantidad", pd.Series([1]*len(df))),
                                   errors="coerce").fillna(1)

    print(f"  ✅ ICBF Medidas: {len(df):,} registros")
    return df


df_icbf = scrape_icbf_medidas(session_main, YEARS)

if not df_icbf.empty:
    ruta = os.path.join(OUTPUT_DIR, "lexdata_icbf_medidas.csv")
    df_icbf.to_csv(ruta, index=False)
    print(f"  💾 Guardado: {ruta}")

df_icbf.head(3)


  ✗ ICBF ID wpqv-gzbz → HTTP 404
  ✗ ICBF ID sgf5-3gg7 → HTTP 404
  ✗ ICBF ID t2uk-ntbr → HTTP 404
  → Buscando alternativa en catálogo Socrata...
    📋 Catálogo: 'Casos atendidos por Comisaría de Familia, Municipio de Susa' | ID: 23x2-bgqa
    ✅ ID 23x2-bgqa responde — usando como alternativa
[ICBF Medidas] Extrayendo — ID: 23x2-bgqa...
  → Columnas (muestra): ['clase_de_atencion', 'a_o_2008', 'a_o_2009', 'a_o_2010', 'a_o_2011', 'a_o_2012', 'a_o_2013', 'a_o_2014']
  → 70 registros crudos
  ✅ ICBF Medidas: 66 registros
  💾 Guardado: C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\data_judicial\lexdata_icbf_medidas.csv


,clase_de_atencion,a_o_2008,a_o_2009,a_o_2010,a_o_2011,a_o_2012,a_o_2013,a_o_2014,a_o_2015,a_o_2016,a_o_2017,a_o_2018,a_o_2019,a_o_2020,a_o_2021,a_o_2022,municipio,tipo_ciclo,fuente,cantidad
0,RECONOCIMIENTO VOLUNTARIO DE PATERNIDAD.,0,0,0,0,0,0,0,0,0,3,7,3,3,2,3,SIN_DATO,MEDIDA_ICBF,ICBF — Medidas de Protección (23x2-bgqa),1.0
1,DEMANDAS POR RECONOCIMIENTO PATERNO O FILIACIÓN.,0,0,0,0,0,0,0,0,0,2,2,1,0,0,1,SIN_DATO,MEDIDA_ICBF,ICBF — Medidas de Protección (23x2-bgqa),1.0
2,ALIMENTOS,0,0,0,0,0,0,0,0,0,37,37,32,31,38,22,SIN_DATO,MEDIDA_ICBF,ICBF — Medidas de Protección (23x2-bgqa),1.0


### Fuente 5 — Directorio de Comisarías

In [10]:
def scrape_comisarias_directorio(session) -> pd.DataFrame:
    """
    Directorio de Comisarías de Familia Ley 2126 (7tuu-upb2).
    FIX v7: normalizar_columnas() + rename seguro para evitar cols duplicadas.
    Columnas reales: nombre (depto), nombre_1 (municipio), c_digo_dane_municipio, ...
    """
    DID = DATASETS["comisarias_directorio"]
    if not DATASETS_DISPONIBLES.get("comisarias_directorio", False):
        print("  ⚠ comisarias_directorio no disponible")
        return pd.DataFrame()

    print("[Directorio Comisarías] Extrayendo...")

    raw = socrata_get(session, DID, {"$limit": 2000})
    print(f"  → {len(raw)} registros totales")
    if not raw:
        return pd.DataFrame()

    df = pd.DataFrame(raw).drop_duplicates()
    print(f"  → Columnas detectadas: {list(df.columns)}")

    # ✅ Mapear 'nombre' → 'departamento' ANTES de normalizar_columnas
    #    (nombre_1 → municipio ya está en COL_ALIAS)
    #    Solo renombrar si 'nombre' existe y 'departamento' no existe todavía
    if "nombre" in df.columns and "departamento" not in df.columns:
        df = df.rename(columns={"nombre": "departamento"})
        print("    🔄 [Comisarías] 'nombre' → 'departamento'")

    # ✅ normalizar_columnas maneja nombre_1 → municipio
    df = normalizar_columnas(df, fuente_log="Comisarías")

    # ✅ Filtrar por departamento en pandas
    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].apply(normalizar_texto)
        df = filtrar_depto(df)

    # ✅ Normalizar municipio
    if "municipio" in df.columns:
        # Si hay columnas duplicadas 'municipio' (por rename multiple), quedarse con la primera
        if isinstance(df.columns, pd.Index) and df.columns.duplicated().any():
            df = df.loc[:, ~df.columns.duplicated()]
            print("    ⚠ Columnas duplicadas detectadas — eliminando duplicados")
        df["municipio"] = df["municipio"].apply(normalizar_texto)
    else:
        df["municipio"] = "SIN_DATO"

    df["fuente"] = "Directorio Comisarías Ley 2126"

    print(f"  ✅ Comisarías: {len(df)} registros en {DEPARTAMENTOS_FILTRO}")
    return df


df_comisarias = scrape_comisarias_directorio(session_main)
if not df_comisarias.empty:
    ruta = os.path.join(OUTPUT_DIR, "lexdata_comisarias_directorio.csv")
    df_comisarias.to_csv(ruta, index=False)
    print(f"  💾 Guardado: {ruta}")
df_comisarias.head(3)


[Directorio Comisarías] Extrayendo...
  → 1249 registros totales
  → Columnas detectadas: ['c_digo_dane_departamento', 'nombre', 'c_digo_dane_municipio', 'nombre_1', 'tipo_municipio_isla_rea_no', 'categoria_municipio', 'comisarias_ley_2126_100_000', 'existencia_de_comisarias', 'nombre_comisaria', 'direcci_n_comisara', 'telefono_de_contacto', 'horario_de_atenci_n', 'longitud_de_ubicaci_n_de', 'latitud_de_ubicaci_n_de_la', 'coordenadas_de_ubicaci_n']
    🔄 [Comisarías] 'nombre' → 'departamento'
    🔄 [Comisarías] 'nombre_1' → 'municipio'
    🗺  Filtro depto: 1248 → 60 registros ['VALLE DEL CAUCA']
  ✅ Comisarías: 60 registros en ['VALLE DEL CAUCA']
  💾 Guardado: C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\data_judicial\lexdata_comisarias_directorio.csv


,c_digo_dane_departamento,departamento,c_digo_dane_municipio,municipio,tipo_municipio_isla_rea_no,categoria_municipio,comisarias_ley_2126_100_000,existencia_de_comisarias,nombre_comisaria,direcci_n_comisara,telefono_de_contacto,horario_de_atenci_n,longitud_de_ubicaci_n_de,latitud_de_ubicaci_n_de_la,coordenadas_de_ubicaci_n,fuente
57,76,VALLE DEL CAUCA,76520,PALMIRA,Municipio,1,4,3,Comisaría de Familia Móvil,Null,321 5813764,Atención de lunes a viernes de 7:00 a.m. a 4:0...,Null,Null,Null,Directorio Comisarías Ley 2126
58,76,VALLE DEL CAUCA,76520,PALMIRA,Municipio,1,4,3,Comisaría de Familia de Rozo,Calle 10 # 11 – 05,321 5813764,"Atención los lunes, miércoles y viernes, de 7:...",3614674,"-76,387,812","3°36'52.8""N 76°23'16.1""W",Directorio Comisarías Ley 2126
59,76,VALLE DEL CAUCA,76520,PALMIRA,Municipio,1,4,3,Comisaría de Familia ubicada en la Casa de Jus...,"Calle 57 # 44 – 02, barrio Caimitos",2859694 – 318 8276379,"Atención de lunes a viernes, de 7:00 a.m. a 4:...",3548598,"-76,315,400","3°32'55.0""N 76°18'55.4""W",Directorio Comisarías Ley 2126


## Sección 5 — Construcción del IVF (Índice de Vulnerabilidad Familiar)

In [11]:
def agregar_por_municipio_año(df: pd.DataFrame, tipo: str) -> pd.DataFrame:
    """
    Agrega casos por municipio y año, devuelve conteo total.
    FIX v8: df.copy() en todas las operaciones — sin SettingWithCopyWarning.
    """
    if df.empty:
        return pd.DataFrame(columns=["municipio", "anio", f"{tipo}_total"])

    df = df.copy()
    if "anio" not in df.columns:
        df["anio"] = 9999
    cols_req = ["municipio", "anio"]

    if "cantidad" in df.columns:
        df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0)
        agg = df.groupby(cols_req, as_index=False)["cantidad"].sum()
        agg = agg.rename(columns={"cantidad": f"{tipo}_total"})
    else:
        agg = df.groupby(cols_req, as_index=False).size()
        agg = agg.rename(columns={"size": f"{tipo}_total"})

    return agg


# Agregar cada fuente
agg_vif_inmlcf     = agregar_por_municipio_año(df_vif_inmlcf, "vif_inmlcf")
agg_vif_policia    = agregar_por_municipio_año(df_vif_policia, "vif_policia")
agg_icbf           = agregar_por_municipio_año(df_icbf, "medidas_proteccion")
agg_inasistencia   = agregar_por_municipio_año(df_inasistencia, "inasistencia")

print(f"  agg_vif_inmlcf:   {len(agg_vif_inmlcf)} filas")
print(f"  agg_vif_policia:  {len(agg_vif_policia)} filas")
print(f"  agg_icbf:         {len(agg_icbf)} filas")
print(f"  agg_inasistencia: {len(agg_inasistencia)} filas")

# Consolidar VIF (suma INMLCF + Policía)
dfs_vif = [d for d in [agg_vif_inmlcf, agg_vif_policia] if not d.empty]
if dfs_vif:
    vif_cols = ["vif_inmlcf_total", "vif_policia_total"]
    df_vif_total = pd.concat(dfs_vif, join="outer").groupby(
        ["municipio", "anio"], as_index=False
    ).sum(numeric_only=True)
    cols_vif = [c for c in vif_cols if c in df_vif_total.columns]
    df_vif_total["vif_total"] = df_vif_total[cols_vif].sum(axis=1)
else:
    df_vif_total = pd.DataFrame(columns=["municipio", "anio", "vif_total"])

# Feature matrix: outer join progresivo
dfs_merge = [df_vif_total[["municipio", "anio", "vif_total"]]]

if not agg_inasistencia.empty:
    dfs_merge.append(agg_inasistencia.rename(columns={"inasistencia_total": "inasistencia_total"}))
if not agg_icbf.empty:
    dfs_merge.append(agg_icbf)

df_matrix = dfs_merge[0]
for d in dfs_merge[1:]:
    df_matrix = df_matrix.merge(d, on=["municipio", "anio"], how="outer")

# Proxy alimentos: 0.75 × VIF (reemplazar con CSJ cuando esté disponible)
if "alimentos_familia_total" not in df_matrix.columns:
    df_matrix["alimentos_familia_total"] = (
        pd.to_numeric(df_matrix.get("vif_total", 0), errors="coerce").fillna(0) * 0.75
    ).round()
    print("⚠ 'alimentos_familia_total' estimado como proxy (0.75 × VIF)")
    print("  → Reemplazar con datos reales de la Rama Judicial cuando esté disponible")

# Rellenar NaN y asegurar tipos numéricos
for col in ["vif_total", "alimentos_familia_total", "medidas_proteccion_total", "inasistencia_total"]:
    if col not in df_matrix.columns:
        df_matrix[col] = 0
    df_matrix[col] = pd.to_numeric(df_matrix[col], errors="coerce").fillna(0)

print(f"\n✅ Feature matrix: {df_matrix.shape[0]} filas · {df_matrix.shape[1]} columnas")
print(f"   Municipios: {df_matrix['municipio'].nunique()}")
print(f"   Años: {sorted(df_matrix['anio'].dropna().unique().tolist())}")

df_matrix.head()


  agg_vif_inmlcf:   182 filas
  agg_vif_policia:  0 filas
  agg_icbf:         1 filas
  agg_inasistencia: 3 filas
⚠ 'alimentos_familia_total' estimado como proxy (0.75 × VIF)
  → Reemplazar con datos reales de la Rama Judicial cuando esté disponible

✅ Feature matrix: 186 filas · 6 columnas
   Municipios: 42
   Años: [2020, 2021, 2022, 2023, 2024, 9999]


,municipio,anio,vif_total,inasistencia_total,medidas_proteccion_total,alimentos_familia_total
0,ALCALA,2020,3.0,0.0,0.0,2.0
1,ALCALA,2021,1.0,0.0,0.0,1.0
2,ALCALA,2022,1.0,0.0,0.0,1.0
3,ALCALA,2023,5.0,0.0,0.0,4.0
4,ALCALA,2024,1.0,0.0,0.0,1.0


In [12]:
def calcular_ivf(df: pd.DataFrame, pesos: dict, pob_dict: dict) -> pd.DataFrame:
    """
    IVF ponderado — Índice de Vulnerabilidad Familiar.
    Dimensiones v8: VIF · Alimentos · Medidas ICBF · Inasistencia Alimentaria
    """
    df = df.copy()

    df["ivf_score_bruto"] = sum(
        df[col].fillna(0) * peso
        for col, peso in pesos.items()
        if col in df.columns
    )

    min_s, max_s = df["ivf_score_bruto"].min(), df["ivf_score_bruto"].max()
    if max_s > min_s:
        df["ivf_score_ponderado"] = (
            (df["ivf_score_bruto"] - min_s) / (max_s - min_s) * 100
        ).round(1)
    else:
        df["ivf_score_ponderado"] = 50.0

    df["poblacion"] = df["municipio"].map(pob_dict).fillna(50000)
    df["ivf_tasa_100k"] = (df["ivf_score_bruto"] / df["poblacion"] * 100000).round(2)

    return df


df_ivf = calcular_ivf(df_matrix, PESOS_IVF, DANE_POB_2024)

# Columnas de resumen — usar las que realmente existen en df_ivf
agg_cols = {"vif_total": "sum", "alimentos_familia_total": "sum",
            "medidas_proteccion_total": "sum", "inasistencia_total": "sum",
            "ivf_score_bruto": "sum", "ivf_score_ponderado": "mean",
            "ivf_tasa_100k": "mean"}
agg_cols_presentes = {k: v for k, v in agg_cols.items() if k in df_ivf.columns}

df_ivf_resumen = (
    df_ivf.groupby("municipio", as_index=False)
    .agg(agg_cols_presentes)
    .sort_values("ivf_score_ponderado", ascending=False)
    .reset_index(drop=True)
)

p75 = df_ivf_resumen["ivf_score_ponderado"].quantile(0.75)
df_ivf_resumen["alerta"] = df_ivf_resumen["ivf_score_ponderado"] >= p75

print(f"✅ IVF calculado para {len(df_ivf_resumen)} municipios")
print(f"   Umbral de alerta (P75): {p75:.1f}")
print(f"   Municipios en alerta: {df_ivf_resumen['alerta'].sum()}")
print()

cols_show = [c for c in ["municipio", "vif_total", "inasistencia_total",
                          "ivf_score_ponderado", "ivf_tasa_100k", "alerta"]
             if c in df_ivf_resumen.columns]
print(df_ivf_resumen[cols_show].head(10).to_string(index=False))


✅ IVF calculado para 42 municipios
   Umbral de alerta (P75): 1.6
   Municipios en alerta: 11

          municipio  vif_total  inasistencia_total  ivf_score_ponderado  ivf_tasa_100k  alerta
               CALI     3246.0                 0.0                70.34         18.138    True
            PALMIRA      533.0                 0.0                11.54         21.406    True
GUADALAJARA DE BUGA      306.0                 0.0                 6.60         31.174    True
            CARTAGO      256.0                 0.0                 5.56         23.190    True
              YUMBO      163.0                 0.0                 3.50         17.196    True
              TULUA      155.0                 0.0                 3.34          8.786    True
       BUENAVENTURA      154.0                 0.0                 3.34          4.400    True
         CANDELARIA      144.0                 0.0                 3.08         17.336    True
            JAMUNDI      123.0                 0.0

## Sección 6 — Exportación de outputs

In [13]:
ruta_matrix = os.path.join(OUTPUT_DIR, "lexdata_co_ocurrencia_IVF_v8.csv")
df_ivf.to_csv(ruta_matrix, index=False)
print(f"💾 Feature matrix → {ruta_matrix}")
print(f"   Filas: {len(df_ivf):,} · Columnas: {list(df_ivf.columns)}")

ruta_resumen = os.path.join(OUTPUT_DIR, "lexdata_ivf_resumen_municipios.csv")
df_ivf_resumen.to_csv(ruta_resumen, index=False)
print(f"💾 Resumen IVF → {ruta_resumen}")

print()
print("=" * 60)
print("REPORTE DE COBERTURA — Pipeline v8")
print("=" * 60)
fuentes = {
    "INMLCF VIF":           df_vif_inmlcf,
    "Policía VIF":          df_vif_policia,
    "Inasistencia alim.":   df_inasistencia,
    "ICBF Medidas":         df_icbf,
    "Comisarías dir.":      df_comisarias,
}
for nombre, df in fuentes.items():
    if df.empty:
        print(f"  ⚠ {nombre:<22} → VACÍO")
    else:
        muns = df["municipio"].nunique() if "municipio" in df.columns else "N/A"
        print(f"  ✅ {nombre:<22} → {len(df):>7,} filas · {muns} municipios")

print("=" * 60)
print(f"  Feature matrix final: {len(df_ivf):,} filas")
print(f"  Municipios con IVF:   {df_ivf_resumen.shape[0]}")
print(f"  En alerta (IVF P75):  {df_ivf_resumen['alerta'].sum()}")
print("=" * 60)


💾 Feature matrix → C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\data_judicial\lexdata_co_ocurrencia_IVF_v8.csv
   Filas: 186 · Columnas: ['municipio', 'anio', 'vif_total', 'inasistencia_total', 'medidas_proteccion_total', 'alimentos_familia_total', 'ivf_score_bruto', 'ivf_score_ponderado', 'poblacion', 'ivf_tasa_100k']
💾 Resumen IVF → C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\data_judicial\lexdata_ivf_resumen_municipios.csv

REPORTE DE COBERTURA — Pipeline v8
  ✅ INMLCF VIF             →   5,856 filas · 41 municipios
  ⚠ Policía VIF            → VACÍO
  ✅ Inasistencia alim.     →       3 filas · 3 municipios
  ✅ ICBF Medidas           →      66 filas · 1 municipios
  ✅ Comisarías dir.        →      60 filas · 42 municipios
  Feature matrix final: 186 filas
  Municipios con IVF:   42
  En alerta (IVF P75):  11


In [14]:
print(len(df_vif_inmlcf))
print(len(df_vif_policia))
print(len(df_icbf))

5856
0
66


## Notas técnicas v8

| Problema (v7) | Solución (v8) |
|---|---|
| INMLCF/Policía: 600k registros nacionales, Valle del Cauca fuera de los primeros 50k | `$where` con columnas reales verificadas en diagnóstico |
| `$where` roto en v6 por nombres incorrectos | `build_where_depto(col_name)` — constructor explícito con el nombre real de la columna |
| Hurto a personas: mide criminalidad urbana genérica, no ciclo familiar | Reemplazado por **inasistencia alimentaria** (Fiscalía) — delito penal directo del nicho |
| ICBF `wpqv-gzbz` permanentemente 404 | Fallback: lista ICBF_IDS_FALLBACK → búsqueda catálogo Socrata |
| `hurto_total` en PESOS_IVF distorsionaba el IVF | `inasistencia_total` — mismo peso 0.10, mayor validez conceptual |

**Coherencia teórica del IVF v8:**
| Dimensión | Peso | Fuente | Tipo |
|---|---|---|---|
| VIF | 0.40 | INMLCF + Policía | Delito/lesión directa |
| Alimentos familia | 0.30 | Proxy CSJ (0.75×VIF) | Civil — ruptura económica |
| Medidas ICBF | 0.20 | ICBF comisarías | Intervención institucional |
| Inasistencia alimentaria | 0.10 | Fiscalía | Penal — incumplimiento familiar |

**Siguiente paso:** ejecutar `lexdata_modelo_predictivo_demo.ipynb` con la feature matrix generada.
